In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework_wk6') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 23:07:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/06 23:07:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
#Question 1:
spark.version

'4.1.1'

In [3]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-03 00:34:50--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:20a6:c00:b:20a5:b140:21, 2600:9000:20a6:2e00:b:20a5:b140:21, 2600:9000:20a6:2400:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:20a6:c00:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  30.1MB/s    in 2.3s    

2026-03-03 00:34:53 (30.1 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [29]:
!wc -l yellow_tripdata_2025-11.parquet

  282291 yellow_tripdata_2025-11.parquet


In [4]:
!uv add pandas

Resolved 106 packages in 475ms                                       
Audited 4 packages in 1ms                                            


In [5]:
!uv add numpy

Resolved 106 packages in 23ms                                        
Audited 4 packages in 0.17ms                                         


In [6]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

In [7]:
# Question 2
# Read in file as a spark dataframe:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [8]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [9]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
# Repartition into 4 partitions and save to parquet. What's the average size of the parquet? (25.6MB)
df = df.repartition(4)

In [11]:
!pwd

/Users/mishadavies/src/DEZCamp_2026/Week_6


In [12]:
df.write.parquet('yellow_tripdata/2025/11/', mode='overwrite')

In [18]:
# Question 3: Count of taxi trips that started on Nov 15 (162,604)

In [13]:
from pyspark.sql.functions import col

In [14]:
# Need to redo this in spark.sql:
count_records = df.filter(col("tpep_pickup_datetime").cast("date") == "2025-11-15").count()
print(count_records)

162604


In [15]:
#3 alternative method
df.withColumn('tpep_pickup_datetime', F.to_date(df.tpep_pickup_datetime)) \
    .filter("tpep_pickup_datetime = '2025-11-15'") \
    .count()

162604

In [16]:
# Question 4: Longest trip in hours (90.6)
# Need to redo this in spark.sql:
longest_trip = df.withColumn("trip_duration_hours", 
                              (F.unix_timestamp(col("tpep_dropoff_datetime")) - 
                               F.unix_timestamp(col("tpep_pickup_datetime"))) / 3600) \
                 .agg({"trip_duration_hours": "max"}) \
                 .collect()[0][0]

print(f"The longest trip duration is: {longest_trip} hours")

[Stage 17:=============================>                            (4 + 4) / 8]

The longest trip duration is: 90.64666666666666 hours


In [26]:
#4 alternative method
df.withColumn("trip_duration_hours", 
                              (F.unix_timestamp(col("tpep_dropoff_datetime")) - 
                               F.unix_timestamp(col("tpep_pickup_datetime"))) / 3600) \
    .withColumn('tpep_pickup_datetime', F.to_date(df.tpep_pickup_datetime)) \
    .groupBy('tpep_pickup_datetime') \
    .max('trip_duration_hours') \
    .orderBy('max(trip_duration_hours)', ascending=False) \
    .limit(1) \
    .show()

[Stage 23:=============================>                            (4 + 4) / 8]

+--------------------+------------------------+
|tpep_pickup_datetime|max(trip_duration_hours)|
+--------------------+------------------------+
|          2025-11-26|       90.64666666666666|
+--------------------+------------------------+



In [ ]:
# Question 5: spark user interface dashboard runs on local port 4040

In [27]:
# Question 6: Need to join Nov_2025 data & zone data 

# Read in zones
df_zones = spark.read.parquet('zones')

In [28]:
# Look at columns for df_zones and taxi data (df)
df_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [29]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [30]:
# Combine data
df_join = df.join(df_zones, df.PULocationID == df_zones.LocationID)

In [31]:
# Turn into table for use with SQL
df_join.createOrReplaceTempView('tb_join')

In [34]:
# Check if it worked
spark.sql("""
SELECT PULocationID, LocationID, Zone FROM tb_join
LIMIT 10
""").show()

+------------+----------+---------+
|PULocationID|LocationID|     Zone|
+------------+----------+---------+
|         196|       196|Rego Park|
|         196|       196|Rego Park|
|         196|       196|Rego Park|
|         196|       196|Rego Park|
|         196|       196|Rego Park|
|         196|       196|Rego Park|
|         196|       196|Rego Park|
|         196|       196|Rego Park|
|         196|       196|Rego Park|
|         196|       196|Rego Park|
+------------+----------+---------+



In [36]:
#Calculate for HQ:
spark.sql("""
SELECT LocationID, Zone, COUNT(*) as count
FROM tb_join
GROUP BY LocationID, Zone
ORDER BY count ASC
""").show()

# Governor's Island...
# Arden Heights
# Eltingville/Annad...

+----------+--------------------+-----+
|LocationID|                Zone|count|
+----------+--------------------+-----+
|       105|Governor's Island...|    1|
|        84|Eltingville/Annad...|    1|
|         5|       Arden Heights|    1|
|       187|       Port Richmond|    3|
|       199|       Rikers Island|    4|
|       111| Green-Wood Cemetery|    4|
|       109|         Great Kills|    4|
|       204|   Rossville/Woodrow|    4|
|         2|         Jamaica Bay|    5|
|       251|         Westerleigh|   12|
|       176|             Oakwood|   14|
|       172|New Dorp/Midland ...|   14|
|       245|       West Brighton|   14|
|        59|        Crotona Park|   14|
|       253|       Willets Point|   15|
|        27|Breezy Point/Fort...|   16|
|       206|Saint George/New ...|   17|
|        30|       Broad Channel|   18|
|       156|     Mariners Harbor|   21|
|       118|Heartland Village...|   22|
+----------+--------------------+-----+
only showing top 20 rows
